# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col, lower, when, regexp_replace, to_timestamp, to_date, coalesce

# Read from Bronze Table

In [0]:
df = spark.table("ecommerce.`01_bronze`.orders")

# Transformation

## Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))

## Filtering data

In [0]:
df = df.filter(col("order_id").isNotNull())
df = df.filter(col("quantity").isNotNull())
df = df.filter(col("product_name").isNotNull())

## Normalization Product Name

In [0]:
df = df.withColumn(
    "product_name",
    when(lower(trim(col("product_name"))).contains("dinara"), "Dinara")
    .when(lower(trim(col("product_name"))).contains("javatis"), "Javatis")
    .when(lower(trim(col("product_name"))).contains("fonose"), "Fonose")
    .when(lower(trim(col("product_name"))).contains("eurika"), "Eurika")
    .when(lower(trim(col("product_name"))).contains("sendigo"), "Sendigo")
    .when(lower(trim(col("product_name"))).contains("rubika"), "Rubika")
    .when(
        lower(trim(col("product_name"))).contains("sambal") |
        lower(trim(col("product_name"))).contains("bilis"),
        "Sambal Mak Jah"
    )
    .when(lower(trim(col("product_name"))).contains("pool"), "Aqua Bot")
    .otherwise("n/a")
)

## Change String to Date

In [0]:
df = df.withColumn(
    "created_time",
    to_date(
        trim(regexp_replace(col("created_time"), r"[\t\n\r]", "")),
        "dd/MM/yyyy HH:mm:ss"
    )
)

## DataType Transformation

In [0]:
df = df.withColumn("order_id", col("order_id").cast("long"))
df = df.withColumn("sku_id", col("sku_id").cast("long"))
df = df.withColumn("sku_quantity_of_return", col("sku_quantity_of_return").cast("double"))
df = df.withColumn("sku_subtotal_before_discount", col("sku_subtotal_before_discount").cast("double"))
df = df.withColumn("sku_subtotal_before_discount", col("sku_subtotal_before_discount").cast("double"))
df = df.withColumn("sku_subtotal_after_discount", col("sku_subtotal_after_discount").cast("double"))
df = df.withColumn("original_shipping_fee", col("original_shipping_fee").cast("double"))

## Normalization Country, State, Shipping Provider Name

In [0]:
df = (
    df.withColumn(
    "country",
    when(col("country").isNull(), "n/a")
    .otherwise("Malaysia")
)
    .withColumn(
        "state",
        when(col("state").isNull(), "n/a")
        .otherwise(col("state")
    )
)
    .withColumn(
        "shipping_provider_name",
        when(col("shipping_provider_name").isNull(), "n/a")
        .otherwise(col("shipping_provider_name")
    )
)
)

# Writing to Silver Table

In [0]:
(
    df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ecommerce.`02_silver`.silver_orders")
)

## Sanity Checks for Silver Table

In [0]:
%sql
select * from ecommerce.`02_silver`.silver_orders